# Bonus: visualization of RT-DETRv2 internals

Inside the tuned RT-DETRv2 (`rtdetr-recipe_step2`, the best single model): the decoder query embeddings of the detections on a sample of val frames, projected to 2D with t-SNE and labelled by class, size, and error category, and the deformable cross-attention of the decoder: where each query samples the feature maps, how this changes over the 6 decoder layers, and how it relates to object size and scene density. W&B logging of curves, configs, and validation frames for every run is listed in `experiments/results.csv` (`wandb_url`).

| # | Section |
| --- | --- |
| 1 | [Setup](#1-setup) |
| 2 | [Queries of the sampled frames](#2-queries-of-the-sampled-frames) |
| 3 | [Query embeddings](#3-query-embeddings) |
| 4 | [Cross-attention of single queries](#4-cross-attention-of-single-queries) |
| 5 | [Cross-attention statistics](#5-cross-attention-statistics) |
| 6 | [Conclusions](#6-conclusions) |

## 1. Setup

Imports, data root from `DATA_ROOT`, the annotations of the adopted split, and the `best.pt` of `rtdetr-recipe_step2` (exists only on the PC). The model runs in eval mode at its training resolution (shorter side 1024 px, `max_size` 1824).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import torch
from decouple import AutoConfig
from IPython.display import display
from tqdm.auto import tqdm

# make the repository root importable when the kernel starts in notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
# decouple locates .env from the caller file, which is a temp file inside a kernel
config = AutoConfig(search_path=PROJECT_ROOT)

import transformers
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from torchvision.io import ImageReadMode, decode_image

from src.data.annotations import load_annotations, load_split_clips
from src.data.classes import CLASS_NAMES
from src.evaluate import DATA_CONFIG_PATH, PREDICTIONS_FILE, RESULTS_PATH, find_run_dir
from src.evaluation.coco_metrics import PREDICTION_COLUMNS
from src.evaluation.failure_analysis import ERROR_CATEGORIES, find_best_f1_threshold, match_detections
from src.inference.torchvision_detectors import MODEL_STATE_KEY, convert_frames_to_inputs, select_device
from src.models.rtdetr import build_rtdetr, predict_query_internals, summarize_query_attention
from src.training.experiment import load_run_config
from src.visualization.frames import plot_query_attention
from src.visualization.plots import plot_embedding_panels
from src.visualization.style import (
    CLASS_COLORS,
    ERROR_CATEGORY_COLORS,
    FEATURE_LEVEL_COLORS,
    SIZE_BUCKET_COLORS,
    set_plot_theme,
)

# hide the checkpoint load report and progress bar of transformers in the saved outputs
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
set_plot_theme()
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

MODEL = "rtdetr-recipe_step2"
SEED = 42
FRAME_STEP = 10
tables = load_annotations(Path(config("DATA_ROOT")), load_split_clips(DATA_CONFIG_PATH))
val_images = tables.images.filter(pl.col("split") == "val")
val_objects = tables.objects.filter(pl.col("split") == "val")
run_dir = PROJECT_ROOT / find_run_dir(RESULTS_PATH, MODEL)
run_config = load_run_config(run_dir)
device = select_device()
detector = build_rtdetr(run_config)
checkpoint = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)
detector.load_state_dict(checkpoint[MODEL_STATE_KEY])
detector = detector.to(device).eval()
pl.DataFrame({"model": [MODEL], "device": [str(device)], "min_size": [detector.min_size], "max_size": [detector.max_size]})

## 2. Queries of the sampled frames

Every 10th val frame of each clip. A query is kept when its best class score reaches the score threshold that maximizes F1 at IoU 0.5 on all val frames (from the saved val predictions of the run), so the points are the detections the model would report. Each kept query is matched to the GT of its frame as in the failure analysis of `02_two_stage.ipynb` to `05_query_based.ipynb` (correct, duplicate, wrong class, poor localisation, background; detections in ignored regions are left out) and gets a size bucket from its box area (COCO ranges, small below 32² px², large from 96² px²).

In [ ]:
saved_predictions = pl.read_parquet(run_dir / PREDICTIONS_FILE)
threshold = find_best_f1_threshold(match_detections(tables, "val", saved_predictions))["threshold"] or 0.0
sampled_images = val_images.filter((pl.col("frame_id") - 1) % FRAME_STEP == 0).sort("image_id")
objects_per_frame = val_objects.group_by("image_id").len("objects_in_frame")
# the sampled frame with the most labelled objects, kept for the single-query figures of section 4
dense_frame = (
    sampled_images.join(objects_per_frame, on="image_id")
    .sort("objects_in_frame", "image_id", descending=[True, False])
    .row(0, named=True)
)
pl.DataFrame(
    {
        "sampled frames": [sampled_images.height],
        "score threshold": [round(threshold, 4)],
        "densest sampled frame": [f"{dense_frame['clip']} frame {dense_frame['frame_id']}"],
        "objects in it": [dense_frame["objects_in_frame"]],
    }
)

In [ ]:
query_tables, embeddings, attention_tables = [], [], []
for image_id, file_path in tqdm(sampled_images.select("image_id", "file_path").iter_rows(), total=sampled_images.height):
    frame = decode_image(file_path, mode=ImageReadMode.RGB)
    internals = predict_query_internals(detector, convert_frames_to_inputs([frame], device)[0])
    kept = internals.queries.filter(pl.col("score") >= threshold)
    query_tables.append(kept.with_columns(image_id=pl.lit(image_id)))
    embeddings.append(internals.embeddings[kept["query"].to_numpy()])
    attention_tables.append(
        summarize_query_attention(internals, kept["query"].to_list()).with_columns(image_id=pl.lit(image_id))
    )
    if image_id == dense_frame["image_id"]:
        dense_internals = internals
kept_queries = pl.concat(query_tables).with_row_index("point")
embedding_matrix = np.concatenate(embeddings)
attention = pl.concat(attention_tables)
pl.DataFrame({"kept queries": [kept_queries.height], "embedding size": [embedding_matrix.shape[1]]})

In [ ]:
sampled_tables = tables._replace(
    images=tables.images.filter(pl.col("split") == "val", pl.col("image_id").is_in(sampled_images["image_id"].to_list()))
)
matched = match_detections(sampled_tables, "val", kept_queries).detections
area = pl.col("w") * pl.col("h")
points = (
    matched.filter(pl.col("category") != "ignored")
    .join(objects_per_frame, on="image_id", how="left")
    .with_columns(
        predicted_class=pl.col("class_id").replace_strict(dict(enumerate(CLASS_NAMES))),
        gt_class=pl.col("best_class_id").replace_strict(dict(enumerate(CLASS_NAMES)), default=None),
        size_bucket=pl.when(area < 32**2).then(pl.lit("small")).when(area < 96**2).then(pl.lit("medium")).otherwise(pl.lit("large")),
        objects_in_frame=pl.col("objects_in_frame").fill_null(0),
    )
    .sort("point")
)
points.group_by("category").len().sort(pl.col("category").cast(pl.Enum(ERROR_CATEGORIES)))

*Takeaway: written from the outputs of the PC run.*

## 3. Query embeddings

The 256-dimensional output of the last decoder layer of every kept query, reduced to 50 dimensions with PCA and projected to 2D with t-SNE ([scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html); [van der Maaten and Hinton, 2008](https://jmlr.org/papers/v9/vandermaaten08a.html)). t-SNE keeps neighbourhoods, not distances ([Wattenberg et al., 2016](https://distill.pub/2016/misread-tsne/)): clusters and their members are meaningful, the distance between clusters and the cluster sizes are not. The class head is a linear layer on this embedding, so classes the model separates well must form separate regions.

In [ ]:
embedding_values = embedding_matrix[points["point"].to_numpy()]
reduced = PCA(n_components=50, random_state=SEED).fit_transform(embedding_values)
projected = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(reduced)
points = points.with_columns(embedding_x=pl.Series(projected[:, 0]), embedding_y=pl.Series(projected[:, 1]))
fig = plot_embedding_panels(
    points,
    [
        ("predicted_class", CLASS_COLORS, "Predicted class"),
        ("category", ERROR_CATEGORY_COLORS, "Error category at IoU 0.5"),
        ("size_bucket", SIZE_BUCKET_COLORS, "Box size"),
    ],
    f"t-SNE of the decoder query embeddings ({points.height} detections, {sampled_images.height} val frames)",
)

*Takeaway: written from the outputs of the PC run.*

In [ ]:
fig = plot_embedding_panels(
    points.filter(pl.col("category").is_in(["correct", "wrong_class"])),
    [
        ("gt_class", CLASS_COLORS, "GT class of correct and wrong-class detections"),
    ],
    "The same projection coloured by the GT class",
)

*Takeaway: written from the outputs of the PC run.*

Quantitative check in the full 256-dimensional space: for each correct detection, the share of its 10 nearest neighbours (Euclidean, other detections of the sample) that are correct detections of the same class, and the class-to-class share of neighbours. Off-diagonal entries are the confusable classes.

In [ ]:
correct = points.filter(pl.col("category") == "correct")
correct_values = embedding_matrix[correct["point"].to_numpy()]
neighbours = NearestNeighbors(n_neighbors=11).fit(correct_values).kneighbors(correct_values, return_distance=False)[:, 1:]
classes = correct["class_id"].to_numpy()
neighbour_classes = classes[neighbours]
neighbour_share = pl.DataFrame(
    [
        {"class": CLASS_NAMES[c], "detections": int((classes == c).sum())}
        | {CLASS_NAMES[o]: float((neighbour_classes[classes == c] == o).mean()) for o in range(len(CLASS_NAMES))}
        for c in range(len(CLASS_NAMES))
        if (classes == c).any()
    ]
)
neighbour_share.with_columns(pl.selectors.float().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    neighbour_share.select(CLASS_NAMES).to_numpy(), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
    xticklabels=CLASS_NAMES, yticklabels=neighbour_share["class"].to_list(), ax=ax,
)
ax.set(xlabel="Class of the neighbour", ylabel="Class of the correct detection", title="Classes of the 10 nearest neighbours in the embedding")
fig.tight_layout()

*Takeaway: written from the outputs of the PC run.*

In [ ]:
points.filter(pl.col("category") == "wrong_class").group_by("gt_class", "predicted_class").len().sort("len", descending=True).head(10)

*Takeaway: written from the outputs of the PC run.*

## 4. Cross-attention of single queries

RT-DETRv2 has no dense attention map in the decoder: its multi-scale deformable cross-attention ([Zhu et al., 2021](https://arxiv.org/abs/2010.04159); RT-DETRv2: [Lv et al., 2024](https://arxiv.org/abs/2407.17140)) lets each query sample 4 locations per head (8 heads) on each of the 3 feature levels (strides 8, 16, 32), offset from its current reference box, and weights them. The panels show these 96 locations per query on the sampled frame with the most labelled objects, coloured by level, with an area proportional to the weight, together with the box of that decoder layer. Queries: the highest-scoring correct detection of each size bucket and the highest-scoring background detection.

In [ ]:
dense_points = points.filter(pl.col("image_id") == dense_frame["image_id"])
example_queries = pl.concat(
    [
        dense_points.filter(pl.col("category") == "correct").sort("score", descending=True).unique("size_bucket", keep="first", maintain_order=True),
        dense_points.filter(pl.col("category") == "background").sort("score", descending=True).head(1),
    ]
)
display(example_queries.select("query", "predicted_class", "score", "category", "size_bucket", "w", "h"))
fig = plot_query_attention(
    Path(dense_frame["file_path"]), dense_internals, example_queries["query"].to_list(), [0, 2, 5], 5.0, 160
)
fig.suptitle(f"Cross-attention sampling locations, {dense_frame['clip']} frame {dense_frame['frame_id']} ({dense_frame['width']}×{dense_frame['height']} px)")
fig.tight_layout(rect=(0, 0.04, 1, 0.97))

*Takeaway: written from the outputs of the PC run.*

## 5. Cross-attention statistics

Per kept query and decoder layer (`summarize_query_attention`): the share of attention on locations inside the final box, the spread of the locations around the box centre relative to the box size, and the attention share of each feature level, averaged over the 8 heads. Correct detections only, by size bucket, then by scene density.

In [ ]:
attention_points = attention.join(
    points.select("image_id", "query", "category", "size_bucket", "predicted_class", "objects_in_frame"),
    on=["image_id", "query"],
)
correct_attention = attention_points.filter(pl.col("category") == "correct").with_columns(decoder_layer=pl.col("layer") + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, label in ((axes[0], "inside_share", "Attention inside the final box"), (axes[1], "spread", "Spread / √(box area)")):
    sns.lineplot(
        correct_attention.select("decoder_layer", "size_bucket", column).to_dict(as_series=False),
        x="decoder_layer", y=column, hue="size_bucket", hue_order=list(SIZE_BUCKET_COLORS),
        palette=SIZE_BUCKET_COLORS, marker="o", errorbar=("pi", 50), ax=ax,
    )
    ax.set(xlabel="Decoder layer", ylabel=label)
axes[1].set_yscale("log")
fig.suptitle("Cross-attention of correct detections by decoder layer (median line, interquartile band)")
fig.tight_layout()

*Takeaway: written from the outputs of the PC run.*

In [ ]:
level_columns = [c for c in attention.columns if c.startswith("level_share_")]
level_table = (
    correct_attention.filter(pl.col("layer") == pl.col("layer").max())
    .group_by("size_bucket").agg(pl.len().alias("detections"), *(pl.col(c).mean().round(3) for c in level_columns))
    .sort(pl.col("size_bucket").cast(pl.Enum(list(SIZE_BUCKET_COLORS))))
)
level_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bottom = np.zeros(level_table.height)
for column, color in zip(level_columns, FEATURE_LEVEL_COLORS, strict=True):
    values = level_table[column].to_numpy()
    ax.bar(level_table["size_bucket"].to_list(), values, bottom=bottom, color=color, label=column.replace("level_share_", "stride "))
    bottom += values
ax.set(xlabel="Box size", ylabel="Share of attention", title="Attention per feature level, last decoder layer, correct detections")
ax.legend()
fig.tight_layout()

*Takeaway: written from the outputs of the PC run.*

In [ ]:
objects = pl.col("objects_in_frame")
last_layer = correct_attention.filter(pl.col("layer") == pl.col("layer").max()).with_columns(
    density=pl.when(objects < 20).then(pl.lit("< 20 objects"))
    .when(objects < 50).then(pl.lit("20 to 49"))
    .when(objects < 100).then(pl.lit("50 to 99"))
    .otherwise(pl.lit("≥ 100"))
    .cast(pl.Enum(["< 20 objects", "20 to 49", "50 to 99", "≥ 100"]))
)
last_layer.group_by("density", "size_bucket").agg(
    pl.len().alias("detections"),
    pl.col("inside_share").median().round(3),
    pl.col("spread").median().round(3),
).sort("size_bucket", "density").pivot("density", index="size_bucket", values=["inside_share", "spread"])

*Takeaway: written from the outputs of the PC run.*

In [ ]:
by_category = attention_points.filter(pl.col("layer") == pl.col("layer").max()).group_by("category").agg(
    pl.len().alias("detections"),
    pl.col("inside_share").median().round(3),
    pl.col("spread").median().round(3),
    *(pl.col(c).mean().round(3) for c in level_columns),
).sort(pl.col("category").cast(pl.Enum(ERROR_CATEGORIES)))
by_category

*Takeaway: written from the outputs of the PC run.*

## 6. Conclusions

*Takeaway: written from the outputs of the PC run.*